<a href="https://colab.research.google.com/github/JulioCesarVilella-Napa/Curriculo/blob/master/APP_gera_RelFotos_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

App com interface original

In [ ]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLineEdit, QLabel, QTextEdit, QProgressBar, QFileDialog,
    QScrollArea, QGridLayout, QSizePolicy, QSpacerItem, QMessageBox
)
from PyQt6.QtGui import QTextCursor, QIcon, QPixmap
from PyQt6.QtCore import QThread, pyqtSignal, Qt
import os
import glob
import datetime

# Novos imports da célula 4fa283f3
# New imports from cell 4fa283f3
import shutil
from PIL import Image
from fpdf import FPDF

# --- Classe Geradora de PDF (adaptada de 4fa283f3) ---
# --- PDF Generator Class (adapted from 4fa283f3) ---
class PDF(FPDF):
    def __init__(self, background_path, pdf_title):
        super().__init__()
        self.background_path = background_path
        self.pdf_title = pdf_title
        self.set_margins(20, 40, 20) # Margens padrão / Standard margins
        self.set_auto_page_break(auto=True, margin=20)
        self.current_y = self.t_margin

    def header(self):
        if self.background_path and os.path.exists(self.background_path):
            self.image(self.background_path, x=0, y=0, w=self.w, h=self.h)
        self.set_font('Arial', 'B', 15)
        self.set_y(10)
        self.cell(0, 10, self.pdf_title, 0, 1, 'R') # Título alinhado à direita / Title aligned right
        self.set_y(self.t_margin)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'L') # Número da página alinhado à esquerda / Page number aligned left

    def get_image_dimensions(self, path):
        try:
            with Image.open(path) as img:
                return img.size
        except FileNotFoundError: # Arquivo não encontrado / File not found
            return (0, 0)
        except Exception as e:
            return (0, 0)

    def add_centered_photo(self, img_path, legend, max_width, max_height):
        if not os.path.exists(img_path):
            return

        original_width, original_height = self.get_image_dimensions(img_path)
        if original_width == 0 or original_height == 0:
            return

        aspect_ratio = original_width / original_height

        actual_width = max_width
        actual_height = actual_width / aspect_ratio

        if actual_height > max_height:
            actual_height = max_height
            actual_width = actual_height * aspect_ratio

        # Garante que a imagem não exceda a largura máxima
        # Ensures the image does not exceed the maximum width
        if actual_width > max_width:
             actual_width = max_width
             actual_height = actual_width / aspect_ratio

        largura_pagina = self.w # Largura da página / Page width
        x_centralizado_img = (largura_pagina - actual_width) / 2 # Posição X para centralizar a imagem / X position to center the image

        space_after_image = 5
        estimated_legend_height = 8
        space_after_legend = 10

        self.set_y(self.current_y)

        self.image(img_path, x=x_centralizado_img, y=self.current_y, w=actual_width, h=actual_height)

        self.current_y += actual_height + space_after_image
        self.set_y(self.current_y)

        self.set_font('Arial', 'B', 11)
        # Altera a largura da célula da legenda para usar a largura total disponível da página,
        # em vez da largura da imagem, garantindo espaço suficiente para uma única linha.
        legend_available_width = self.w - self.l_margin - self.r_margin
        self.set_x(self.l_margin)
        self.multi_cell(legend_available_width, estimated_legend_height, legend, 0, 'C')

        self.current_y = self.get_y() + space_after_legend

class Worker(QThread):
    progress = pyqtSignal(int)
    log_msg = pyqtSignal(str)
    finished = pyqtSignal()

    def __init__(self, photos_data, title, bg_path, output_file):
        super().__init__()
        self.photos_data = photos_data # List of {'path': path, 'caption': caption}
        self.title = title
        self.bg_path = bg_path
        self.output_file = output_file

    def run(self):
        try:
            pdf_final = PDF(self.bg_path, self.title)

            # Lógica de paginação adaptada de 4fa283f3
            # Pagination logic adapted from 4fa283f3
            PHOTOS_ON_FIRST_PAGE = 2
            PHOTOS_ON_SECOND_PAGE = 2
            PHOTOS_ON_SUBSEQUENT_PAGES = 3

            photo_layout_by_page = []
            num_total_photos = len(self.photos_data)
            current_photo_idx = 0

            if num_total_photos > 0:
                page_photos = []
                for _ in range(min(PHOTOS_ON_FIRST_PAGE, num_total_photos - current_photo_idx)):
                    page_photos.append(current_photo_idx)
                    current_photo_idx += 1
                if page_photos:
                    photo_layout_by_page.append(page_photos)

            if num_total_photos > current_photo_idx:
                page_photos = []
                for _ in range(min(PHOTOS_ON_SECOND_PAGE, num_total_photos - current_photo_idx)):
                    page_photos.append(current_photo_idx)
                    current_photo_idx += 1
                if page_photos:
                    photo_layout_by_page.append(page_photos)

            while num_total_photos > current_photo_idx:
                page_photos = []
                for _ in range(min(PHOTOS_ON_SUBSEQUENT_PAGES, num_total_photos - current_photo_idx)):
                    page_photos.append(current_photo_idx)
                    current_photo_idx += 1
                if page_photos:
                    photo_layout_by_page.append(page_photos)

            SPACE_AFTER_IMAGE = 5
            ESTIMATED_LEGEND_HEIGHT = 8
            SPACE_AFTER_LEGEND = 10
            MIN_PHOTO_HEIGHT = 20
            MAX_PHOTO_WIDTH = 170

            total_processed_photos = 0
            for page_number, photo_indices_on_page in enumerate(photo_layout_by_page):
                pdf_final.add_page()
                pdf_final.current_y = pdf_final.t_margin

                num_photos_on_page = len(photo_indices_on_page)
                available_height = pdf_final.h - pdf_final.t_margin - pdf_final.b_margin

                total_fixed_overhead = num_photos_on_page * (SPACE_AFTER_IMAGE + ESTIMATED_LEGEND_HEIGHT + SPACE_AFTER_LEGEND)

                available_height_for_photos = max(0, available_height - total_fixed_overhead)

                max_height_per_photo = max(MIN_PHOTO_HEIGHT, available_height_for_photos / num_photos_on_page if num_photos_on_page > 0 else MIN_PHOTO_HEIGHT)

                for photo_idx_in_data in photo_indices_on_page:
                    photo_data_entry = self.photos_data[photo_idx_in_data]
                    img_path = photo_data_entry['path']
                    legend = photo_data_entry['caption']

                    self.log_msg.emit(f"Adicionando foto à página {page_number + 1}: {os.path.basename(img_path)} com legenda '{legend}'") # Adding photo to page / Adicionando foto à página
                    pdf_final.add_centered_photo(img_path, legend, MAX_PHOTO_WIDTH, max_height_per_photo)
                    total_processed_photos += 1
                    self.progress.emit(int(100 * total_processed_photos / num_total_photos))

            # Garante que o diretório de saída exista
            # Ensures the output directory exists
            output_dir = os.path.dirname(self.output_file)
            if output_dir and not os.path.exists(output_dir):
                os.makedirs(output_dir, exist_ok=True)

            pdf_final.output(self.output_file)
            self.log_msg.emit(f'PDF gerado com sucesso em: {self.output_file}') # PDF generated successfully at / PDF gerado com sucesso em
            self.progress.emit(100)
        except Exception as e:
            self.log_msg.emit(f'Erro na geração do PDF: {str(e)}') # Error in PDF generation / Erro na geração do PDF
            self.progress.emit(0) # Resetar progresso em caso de erro / Reset progress on error
        self.finished.emit()


class PhotoWidget(QWidget):
    """Um widget personalizado para exibir uma miniatura de foto e seu índice."""
    """A custom widget to display a photo thumbnail and its index."""
    def __init__(self, index, image_path, parent=None):
        super().__init__(parent)
        self.setFixedSize(120, 150) # Tamanho fixo para o widget da foto / Fixed size for the photo widget
        self.layout = QVBoxLayout(self)
        self.layout.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.layout.setContentsMargins(0, 0, 0, 0) # Remover margens para aproximar elementos / Remove margins to bring elements closer
        self.layout.setSpacing(2) # Reduzir espaçamento entre elementos / Reduce spacing between elementos

        self.image_label = QLabel()
        self.image_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.image_label.setFixedSize(112, 112) # Tamanho fixo para a miniatura (75% de 150x150) / Fixed size for the thumbnail (75% of 150x150)

        pixmap = QPixmap(image_path)
        if not pixmap.isNull():
            scaled_pixmap = pixmap.scaled(112, 112, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation)
            self.image_label.setPixmap(scaled_pixmap)
        else:
            self.image_label.setText("Imagem Inválida") # Invalid Image

        self.index_label = QLabel(f"Foto {index}") # Photo {index}
        self.index_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.index_label.setStyleSheet("font-weight: bold;")

        self.layout.addWidget(self.image_label)
        self.layout.addWidget(self.index_label)


class MainWindowV2(QMainWindow):
    # restart_requested = pyqtSignal() # Sinal para solicitar reinício do aplicativo - REMOVED

    def __init__(self):
        super().__init__()
        self.setWindowTitle('Relatório Fotográfico - Versão Galeria') # Photo Report - Gallery Version
        self.resize(400, 700) # Tamanho inicial menor, similar a um celular / Smaller initial size, similar to a mobile phone
        self.folder_path = None
        # --- ALTERAÇÃO AQUI: Definir o caminho da imagem de fundo permanentemente ---
        # --- CHANGE HERE: Set the background image path permanently ---
        self.bg_path = 'caminho/para/sua/imagem_de_fundo.png' # <<< SUBSTITUA ESTE CAMINHO PELO DA SUA IMAGEM DE FUNDO REAL
                                                            # <<< REPLACE THIS PATH WITH YOUR ACTUAL BACKGROUND IMAGE PATH

        self.photos = [] # Stores {'path': path, 'caption': initial_default_caption}
        self.default_initial_legends = [
            "Logradouro - Lado Direito", # Street - Right Side
            "Logradouro - Lado Esquerdo", # Street - Left Side
            "Fachada do Imóvel", # Property Facade
            "Numeração do Imóvel", # Property Numbering
            "Detalhes do Imóvel" # Property Details
        ]

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        # Seleção de pasta
        # Folder selection
        hl_folder = QHBoxLayout()
        hl_folder.addWidget(QLabel('Pasta:')) # Folder:
        self.folder_label = QLabel('Nenhuma pasta selecionada') # No folder selected
        hl_folder.addWidget(self.folder_label)
        hl_folder.addStretch()
        self.folder_btn = QPushButton('Selecionar Pasta') # Select Folder
        self.folder_btn.clicked.connect(self.select_folder)
        hl_folder.addWidget(self.folder_btn)
        layout.addLayout(hl_folder)

        # Título
        # Title
        layout.addWidget(QLabel('Título do PDF:')) # PDF Title:
        self.title_edit = QLineEdit('Relatório Fotográfico') # Photo Report
        layout.addWidget(self.title_edit)

        # Imagem de fundo
        # Background image
        hl_bg = QHBoxLayout()
        hl_bg.addWidget(QLabel('Fundo:')) # Background:
        # --- ALTERAÇÃO AQUI: Exibir o caminho da imagem de fundo permanente e remover o botão de seleção ---
        # --- CHANGE HERE: Display the permanent background image path and remove the selection button ---
        if self.bg_path and os.path.exists(self.bg_path):
            self.bg_label = QLabel(os.path.basename(self.bg_path)) # Display filename if path is valid
        else:
            self.bg_label = QLabel('Fundo fixo (arquivo não encontrado ou não definido)') # Indicate fixed background

        hl_bg.addWidget(self.bg_label)
        # hl_bg.addWidget(self.bg_btn) # REMOVIDO: Botão de seleção de fundo
        layout.addLayout(hl_bg)

        # Área de rolagem para as fotos (substitui a tabela)
        # Scroll area for photos (replaces table)
        self.scroll_area = QScrollArea()
        self.scroll_area.setWidgetResizable(True)

        self.photos_grid_widget = QWidget() # Widget container para o grid / Widget container for the grid
        # Set size policy for the container widget to expand horizontally and vertically
        # Define a política de tamanho para o widget contêiner para expandir horizontal e verticalmente
        self.photos_grid_widget.setSizePolicy(QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Expanding)
        self.photos_grid_layout = QGridLayout(self.photos_grid_widget)
        self.photos_grid_layout.setAlignment(Qt.AlignmentFlag.AlignLeft | Qt.AlignmentFlag.AlignTop)
        # Add spacing between items in the grid for better visual separation
        # Adiciona espaçamento entre os itens na grade para melhor separação visual
        self.photos_grid_layout.setHorizontalSpacing(10)
        self.photos_grid_layout.setVerticalSpacing(10)

        self.scroll_area.setWidget(self.photos_grid_widget)

        layout.addWidget(self.scroll_area)

        # Entrada numérica para ordem das fotos
        # Numerical input for photo order
        hl_order_input = QHBoxLayout()
        hl_order_input.addWidget(QLabel('Ordem das Fotos (Ex: 3,4,1):')) # Photo Order (Ex: 3,4,1):
        self.order_input = QLineEdit()
        self.order_input.setPlaceholderText('Entre com os números das fotos separados por vírgula') # Enter photo numbers separated by commas
        hl_order_input.addWidget(self.order_input)
        layout.addLayout(hl_order_input)

        # Botão Gerar PDF
        # Generate PDF Button
        hl_generate = QHBoxLayout()
        hl_generate.addStretch()
        self.gen_btn = QPushButton('Gerar PDF') # Generate PDF
        self.gen_btn.clicked.connect(self.generate_pdf)
        hl_generate.addWidget(self.gen_btn)
        layout.addLayout(hl_generate)

        # Progresso
        # Progress
        self.progress = QProgressBar()
        self.progress.setRange(0, 100)
        self.progress.setValue(0)
        layout.addWidget(self.progress)

        # Log
        # Log
        layout.addWidget(QLabel('Log:')) # Log:
        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setFixedHeight(100) # Reduzir a altura da caixa de log / Reduce log box height
        layout.addWidget(self.log)

        # Flag para controlar o reinício, acessível pelo loop principal
        self.restart_flag = False

    def _get_available_grid_width(self):
        # Get the width of the viewport inside the scroll area
        # Obtém a largura da viewport dentro da área de rolagem
        viewport_width = self.scroll_area.viewport().width()
        # Subtract the width of the vertical scrollbar if present
        # Subtrai a largura da barra de rolagem vertical, se presente
        if self.scroll_area.verticalScrollBar().isVisible():
            viewport_width -= self.scroll_area.verticalScrollBar().width()
        # Also subtract margins of the QGridLayout itself
        # Subtrai também as margens do próprio QGridLayout
        margins = self.photos_grid_layout.contentsMargins()
        viewport_width -= margins.left() + margins.right()
        return max(1, viewport_width) # Ensure it's at least 1 / Garante que seja no mínimo 1

    def clear_layout(self, layout):
        """Limpa todos os widgets de um layout."""
        """Clears all widgets from a layout."""
        if layout is not None:
            while layout.count():
                item = layout.takeAt(0)
                widget = item.widget()
                if widget is not None:
                    widget.deleteLater()
                else:
                    self.clear_layout(item.layout())

    def _update_photos_grid_layout(self):
        self.clear_layout(self.photos_grid_layout) # Clear existing widgets / Limpa widgets existentes

        if not self.photos:
            return

        # Calculate number of columns dynamically
        # Calcula o número de colunas dinamicamente
        widget_fixed_width = 120 # From PhotoWidget.setFixedSize(120, ...) / Do PhotoWidget.setFixedSize(120, ...)
        horizontal_spacing = self.photos_grid_layout.horizontalSpacing() # Get spacing from layout / Obtém espaçamento do layout
        item_total_width_estimate = widget_fixed_width + horizontal_spacing

        available_width = self._get_available_grid_width()

        num_columns = max(1, int(available_width / item_total_width_estimate))

        current_row = 0
        current_col = 0

        for i, photo_entry in enumerate(self.photos):
            photo_widget = PhotoWidget(i + 1, photo_entry['path']) # Passar índice 1-based e caminho / Pass 1-based index and path
            self.photos_grid_layout.addWidget(photo_widget, current_row, current_col)

            current_col += 1
            if current_col >= num_columns:
                current_col = 0
                current_row += 1

        # Add horizontal spacer to the last row to push items to the left
        # Adiciona um espaçador horizontal na última linha para empurrar os itens para a esquerda
        if current_col > 0: # If there are items in the last row / Se houver itens na última linha
            spacer = QSpacerItem(0, 0, QSizePolicy.Policy.Expanding, QSizePolicy.Policy.Minimum)
            self.photos_grid_layout.addItem(spacer, current_row, current_col, 1, num_columns - current_col)

        # Add a vertical spacer to push all content to the top
        # Adiciona um espaçador vertical para empurrar todo o conteúdo para o topo
        vertical_spacer = QSpacerItem(0, 0, QSizePolicy.Policy.Minimum, QSizePolicy.Policy.Expanding)
        # If the grid is empty or only has one row, place spacer below the potential content
        # Se a grade estiver vazia ou tiver apenas uma linha, coloca o espaçador abaixo do conteúdo potencial
        if current_row == 0 and num_columns > 0: # No photos or only a single row, ensure spacer is at least below 0,0 / Nenhuma foto ou apenas uma linha, garante que o espaçador esteja pelo menos abaixo de 0,0
            self.photos_grid_layout.addItem(vertical_spacer, 1, 0, 1, num_columns)
        elif current_row > 0: # More than one row, place spacer below the last row / Mais de uma linha, coloca o espaçador abaixo da última linha
            self.photos_grid_layout.addItem(vertical_spacer, current_row + 1, 0, 1, num_columns)


    def select_folder(self):
        path = QFileDialog.getExistingDirectory(self, 'Selecionar Pasta de Fotos') # Select Photo Folder
        if path:
            self.folder_path = path
            self.folder_label.setText(path)
            self.load_photos()

    def load_photos(self):
        if not self.folder_path:
            return
        exts = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']
        images = []
        for ext in exts:
            images.extend(glob.glob(os.path.join(self.folder_path, ext)))
        images.sort() # Ordenar as imagens por nome de arquivo / Sort images by file name

        new_photos_data = []
        for i, img_path in enumerate(images):
            caption = ""
            # A legenda inicial não é mais atribuída aqui para exibição,
            # apenas para manter a estrutura de dados original, se necessário futuramente.
            # The initial caption is no longer assigned here for display,
            # only to maintain the original data structure, if necessary in the future.
            new_photos_data.append({'path': img_path, 'caption': caption})

        self.photos = new_photos_data
        self._update_photos_grid_layout() # Call the new update method / Chama o novo método de atualização
        self.log_message(f'Carregadas {len(images)} fotos da pasta {self.folder_path}') # Loaded {len(images)} photos from folder {self.folder_path}

    # Removed populate_photos_grid as its logic is now in _update_photos_grid_layout

    def resizeEvent(self, event):
        super().resizeEvent(event)
        # Only update the grid if photos are loaded to avoid unnecessary processing
        # Atualiza a grade apenas se as fotos estiverem carregadas para evitar processamento desnecessário
        if self.photos:
            self._update_photos_grid_layout()

    # def select_bg(self):
    #     file, _ = QFileDialog.getOpenFileName(self, 'Imagem de Fundo', '', 'Imagens (*.jpg *.jpeg *.png)') # Background Image
    #     if file:
    #         self.bg_path = file
    #         self.bg_label.setText(os.path.basename(file))

    def generate_pdf(self):
        # 1. Parse user's numerical order input
        # 1. Analisa a entrada numérica de ordem do usuário
        order_text = self.order_input.text().strip()
        if not order_text:
            QMessageBox.warning(self, 'Aviso', 'Por favor, insira a ordem das fotos (números separados por vírgula).') # Warning: Please enter photo order (numbers separated by commas).
            return

        selected_original_indices = []
        try:
            # Convert 1-based user input to 0-based list indices
            # Converte a entrada do usuário baseada em 1 para índices de lista baseados em 0
            selected_indices_1_based = [int(idx.strip()) for idx in order_text.split(',') if idx.strip()]
            for idx_1_based in selected_indices_1_based:
                idx_0_based = idx_1_based - 1
                if not (0 <= idx_0_based < len(self.photos)):
                    QMessageBox.warning(self, 'Erro', f'Número de foto inválido: {idx_1_based}. Por favor, insira números entre 1 e {len(self.photos)}.') # Error: Invalid photo number: {idx_1_based}. Please enter numbers between 1 and {len(self.photos)}.
                    return
                selected_original_indices.append(idx_0_based)
        except ValueError:
            QMessageBox.warning(self, 'Erro', 'Entrada de ordem inválida. Use números separados por vírgula (Ex: 3,4,1).') # Error: Invalid order input. Use numbers separated by commas (Ex: 3,4,1).
            return

        # 2. Construct the photos_data list for the Worker based on the chosen order
        # 2. Constrói a lista photos_data para o Worker com base na ordem escolhida
        photos_data_for_worker = []
        for i, original_photo_idx in enumerate(selected_original_indices):
            photo_entry = self.photos[original_photo_idx] # Get the original photo entry / Obtém a entrada da foto original
            # Assign caption based on its position in the *newly ordered list for the PDF*
            # Atribui legenda com base em sua posição na *lista recém-ordenada para o PDF*
            caption = ""
            if i < len(self.default_initial_legends):
                caption = self.default_initial_legends[i]
            else:
                caption = "Detalhes do Imóvel" # Default for photos beyond the initial set / Padrão para fotos além do conjunto inicial
            photos_data_for_worker.append({'path': photo_entry['path'], 'caption': caption})

        if not photos_data_for_worker:
            QMessageBox.warning(self, 'Aviso', 'Nenhuma foto selecionada na ordem fornecida.') # Warning: No photos selected in the provided order.
            return

        # 3. Proceed with PDF generation using photos_data_for_worker
        # 3. Procede com a geração do PDF usando photos_data_for_worker
        timestamp = datetime.datetime.now().strftime("_%Y%m%d_%H%M%S")
        suggested_filename = os.path.join(self.folder_path or '', f'relatorio_{timestamp}.pdf') # report_{timestamp}.pdf

        output_file, _ = QFileDialog.getSaveFileName(self, 'Salvar PDF', suggested_filename, 'PDF (*.pdf)') # Save PDF
        if not output_file:
            return

        self.gen_btn.setEnabled(False)
        self.progress.setValue(0)
        self.worker = Worker(photos_data_for_worker, self.title_edit.text(), self.bg_path, output_file) # Pass the newly constructed list / Passa a lista recém-construída
        self.worker.progress.connect(self.progress.setValue)
        self.worker.log_msg.connect(self.log_message)
        self.worker.finished.connect(self.on_worker_finished)
        self.worker.start()

    def log_message(self, msg):
        cursor = self.log.textCursor()
        cursor.movePosition(QTextCursor.MoveOperation.End)
        self.log.setTextCursor(cursor)
        self.log.insertPlainText(msg + '\n')
        self.log.ensureCursorVisible()

    def on_worker_finished(self):
        self.gen_btn.setEnabled(True)
        self.progress.setValue(100)
        self.log_message('Processamento concluído.') # Processing complete.

        # Pergunta ao usuário se deseja reiniciar o aplicativo
        reply = QMessageBox.question(self, 'PDF Gerado', 'PDF gerado com sucesso! Deseja reiniciar o aplicativo?',
                                     QMessageBox.StandardButton.Yes | QMessageBox.StandardButton.No,
                                     QMessageBox.StandardButton.No)

        if reply == QMessageBox.StandardButton.Yes:
            self.restart_flag = True # Define a flag para indicar reinício
            self.close() # Fecha a janela
            QApplication.instance().exit(0) # Termina o loop de eventos para reiniciar
        else:
            self.restart_flag = False # Define a flag como False para indicar encerramento
            self.close() # Fecha a janela
            QApplication.instance().quit() # Termina a aplicação completamente

    def closeEvent(self, event):
        # Este método é chamado quando a janela é fechada (e.g., pelo botão 'X')
        if not self.restart_flag:
            # Se não estamos reiniciando, garantimos que a aplicação PyQt seja encerrada.
            QApplication.instance().quit()
        event.accept() # Aceita o evento de fechamento, permitindo que a janela feche


# Para executar esta nova versão, descomente as linhas abaixo:
# To run this new version, uncomment the lines below:
if __name__ == '__main__':
    app = QApplication.instance()
    if app is None:
        app = QApplication([])
        # É importante definir setQuitOnLastWindowClosed(False)
        # para que a instância do QApplication não se encerre
        # completamente ao fechar a janela, permitindo o reinício.
        app.setQuitOnLastWindowClosed(False)

    while True:
        win = MainWindowV2()
        win.show()
        # Bloqueia aqui até que a janela seja fechada e QApplication.exit() ou QApplication.quit() seja chamado
        app.exec()

        if win.restart_flag:
            print("Reiniciando o aplicativo...")
            # O loop continuará para criar uma nova instância de MainWindowV2.
            # A instância antiga de `win` será descartada e coletada pelo garbage collector.
            # A instância de QApplication persiste.
        else:
            print("Aplicativo encerrado normalmente.")
            break # Sai do loop se o reinício não foi solicitado (ou seja, o usuário fechou a janela diretamente)